# 🌍 Project 39 — Land Cover Change Matrix

## 1. Introduce the Project

In Project 38, we asked:

> **Where did non-urban land become urban?**

Now we're going one step further.

Instead of looking at only urbanization, we want to know **all important land-cover transitions** between two years.

For example:

```text
2020                    2021

Forest      ─────────→  Agriculture
Forest      ─────────→  Urban
Water       ─────────→  Cropland
Agriculture ─────────→  Urban
```

This is called a **Land Cover Change Matrix** or **transition matrix**.

It helps us answer:

> **What type of land changed into what other type of land?**

---

# 2. Big Picture Workflow

```text
       Land Cover 2020
              │
              │
              ▼
       Pixel comparison
              ▲
              │
              │
       Land Cover 2021
              │
              ▼
       Identify transitions
              │
      ┌───────┼────────┐
      ▼       ▼        ▼
   Forest   Forest    Water
      ↓       ↓        ↓
    Crop    Urban    Crop
      │       │        │
      └───────┼────────┘
              ▼
       Calculate area
              │
              ▼
       Change matrix
```

---

# 3. Main Concept — What Is a Change Matrix?

Imagine we have:

### 2020

```text
Forest
Forest
Agriculture
Water
```

### 2021

```text
Agriculture
Urban
Urban
Cropland
```

We can compare each pixel:

```text
2020          2021

Forest    →   Agriculture
Forest    →   Urban
Agriculture → Urban
Water     →   Cropland
```

We then count how much area experienced each transition.

---

# 4. Land Cover Classes

We'll continue using **ESA WorldCover**.

Some important classes are:

| Class | Land cover             |
| ----: | ---------------------- |
|    10 | Tree cover             |
|    20 | Shrubland              |
|    30 | Grassland              |
|    40 | Cropland               |
|    50 | Built-up               |
|    60 | Bare/sparse vegetation |
|    70 | Snow/ice               |
|    80 | Permanent water        |
|    90 | Herbaceous wetland     |
|    95 | Mangroves              |
|   100 | Moss/lichen            |

For this project, we'll focus mainly on:

```text
10 → Forest / tree cover
40 → Cropland
50 → Built-up
80 → Water
```

**Small terminology note:** WorldCover's class 10 is officially **Tree Cover**, not necessarily "forest." For this beginner exercise, we'll use "forest/tree cover" as shorthand, but research analysis should preserve the official class definition.

---

# 5. The Key Idea: Compare Two Pixels

Suppose one pixel has:

```text
2020 = 10
2021 = 40
```

Then:

```text
Tree cover → Cropland
```

Another:

```text
2020 = 10
2021 = 50
```

means:

```text
Tree cover → Built-up
```

Another:

```text
2020 = 80
2021 = 40
```

means:

```text
Water → Cropland
```

So we're comparing **the class at the same location** in two different years.

---

# 6. Step 1 — Define the Study Area

We'll use the same Frankfurt-area study region:

```javascript 
var studyArea = ee.Geometry.Polygon([
  [
    [8.35, 50.00],
    [8.90, 50.00],
    [8.90, 50.35],
    [8.35, 50.35],
    [8.35, 50.00]
  ]
]);
```

---

# 7. Step 2 — Load the Two Land-Cover Maps

```javascript 
var worldcover2020 = ee.ImageCollection("ESA/WorldCover/v100")
  .first();

var worldcover2021 = ee.ImageCollection("ESA/WorldCover/v200")
  .first();
```

Then clip them:

```javascript 
var lc2020 = worldcover2020.clip(studyArea);
var lc2021 = worldcover2021.clip(studyArea);
```

Now we have:

```text
lc2020 → land cover at each pixel in 2020
lc2021 → land cover at each pixel in 2021
```

---

# 8. Step 3 — Create a Transition

Let's start with:

> **Tree cover → Cropland**

We need two conditions:

```text
2020 = Tree cover
AND
2021 = Cropland
```

In GEE:

```javascript 
var forestToCrop = lc2020.eq(10)
  .and(lc2021.eq(40));
```

The result is a binary image:

```text
1 → Tree cover → Cropland
0 → Everything else
```

---

# 9. Step 4 — Find Forest → Urban

Same idea:

```javascript
var forestToUrban = lc2020.eq(10)
  .and(lc2021.eq(50));
```

This means:

```text
2020 = Tree cover
AND
2021 = Built-up
```

---

# 10. Step 5 — Find Water → Cropland

```javascript 
var waterToCrop = lc2020.eq(80)
  .and(lc2021.eq(40));
```

So we're specifically looking for:

```text
Water → Cropland
```

---

# 11. Step 6 — Calculate the Area

We've already learned this in Project 11.

First:

```javascript 
var pixelArea = ee.Image.pixelArea();
```

Then multiply by the transition mask:

```javascript 
var forestToCropArea = pixelArea
  .updateMask(forestToCrop);
```

Now calculate the total:

```javascript 
var forestToCropAreaM2 = forestToCropArea.reduceRegion({
  reducer: ee.Reducer.sum(),
  geometry: studyArea,
  scale: 10,
  maxPixels: 1e9
});
```

The result is in **square meters**.

Convert it to km²:

```javascript 
var forestToCropKm2 = ee.Number(
  forestToCropAreaM2.get("area")
).divide(1e6);
```

However, there is a small issue: `pixelArea()` keeps its band name as `"area"`, so the dictionary key is `"area"`.

This is a useful pattern to remember:

```text
Mask
 ↓
pixelArea()
 ↓
updateMask()
 ↓
reduceRegion(sum)
 ↓
Area
```

---

# 12. A Cleaner Way to Calculate Transition Areas

Because we'll calculate several transitions, let's create a function.

```javascript 
function calculateArea(mask) {
  var area = ee.Image.pixelArea()
    .updateMask(mask)
    .reduceRegion({
      reducer: ee.Reducer.sum(),
      geometry: studyArea,
      scale: 10,
      maxPixels: 1e9
    });

  return ee.Number(area.get("area")).divide(1e6);
}
```

Now instead of repeating the entire calculation, we can simply write:

```javascript 
var forestToCropArea = calculateArea(forestToCrop);
```

and:

```javascript 
var forestToUrbanArea = calculateArea(forestToUrban);
```

and:

```javascript 
var waterToCropArea = calculateArea(waterToCrop);
```

### New concept: function

This is exactly the same programming idea you learned in Python:

> Put repeated logic inside a function so you don't have to write it again.

---

# 13. Step 7 — Print the Statistics

```javascript 
print("Forest → Cropland (km²)", forestToCropArea);

print("Forest → Urban (km²)", forestToUrbanArea);

print("Water → Cropland (km²)", waterToCropArea);
```

Now the Console will show the area of each transition.

---

# 14. Step 8 — Display the Transitions

We can also put them on the map.

For example:

```javascript 
Map.addLayer(
  forestToUrban.selfMask(),
  {},
  "Forest → Urban"
);
```

And:

```javascript 
Map.addLayer(
  forestToCrop.selfMask(),
  {},
  "Forest → Cropland"
);
```

And:

```javascript 
Map.addLayer(
  waterToCrop.selfMask(),
  {},
  "Water → Cropland"
);
```

Now you can see **where** the transitions happened, while the statistics tell you **how much area** changed.

---

# 15. What Does a Change Matrix Look Like?

Conceptually, a change matrix looks like this:

| 2020 ↓ / 2021 → |     Tree cover |      Cropland |       Built-up |          Water |
| --------------- | -------------: | ------------: | -------------: | -------------: |
| Tree cover      |      unchanged | Forest → Crop | Forest → Urban | Forest → Water |
| Cropland        |  Crop → Forest |     unchanged |   Crop → Urban |   Crop → Water |
| Built-up        | Urban → Forest |  Urban → Crop |      unchanged |  Urban → Water |
| Water           | Water → Forest |  Water → Crop |  Water → Urban |      unchanged |

The **rows** represent the initial year.

The **columns** represent the later year.

For example:

```text
Row = Tree cover
Column = Built-up
```

means:

> Tree cover → Built-up

This is the basic structure of a land-cover transition matrix.

---

# 16. Why Is This Useful?

A simple land-cover map tells us:

> What is here?

A change map tells us:

> What changed?

A change matrix tells us:

> **What changed into what, and how much?**

For example:

```text
Tree cover → Cropland      12 km²
Tree cover → Built-up       5 km²
Cropland → Built-up         8 km²
Water → Cropland            1 km²
```

This gives us a much more informative description of land-cover change.

---

# 17. Complete Code

Here is the complete beginner-friendly version:

```javascript
// ==========================================
// Project 39 — Land Cover Change Matrix
// ==========================================

// 1. Define study area
var studyArea = ee.Geometry.Polygon([
  [
    [8.35, 50.00],
    [8.90, 50.00],
    [8.90, 50.35],
    [8.35, 50.35],
    [8.35, 50.00]
  ]
]);

// 2. Load WorldCover 2020
var worldcover2020 = ee.ImageCollection("ESA/WorldCover/v100")
  .first();

// 3. Load WorldCover 2021
var worldcover2021 = ee.ImageCollection("ESA/WorldCover/v200")
  .first();

// 4. Clip to study area
var lc2020 = worldcover2020.clip(studyArea);
var lc2021 = worldcover2021.clip(studyArea);


// ==========================================
// Create transition masks
// ==========================================

// Tree cover → Cropland
var forestToCrop = lc2020.eq(10)
  .and(lc2021.eq(40));

// Tree cover → Built-up
var forestToUrban = lc2020.eq(10)
  .and(lc2021.eq(50));

// Water → Cropland
var waterToCrop = lc2020.eq(80)
  .and(lc2021.eq(40));


// ==========================================
// Function to calculate area
// ==========================================

function calculateArea(mask) {

  var area = ee.Image.pixelArea()
    .updateMask(mask)
    .reduceRegion({
      reducer: ee.Reducer.sum(),
      geometry: studyArea,
      scale: 10,
      maxPixels: 1e9
    });

  return ee.Number(area.get("area")).divide(1e6);
}


// ==========================================
// Calculate transition areas
// ==========================================

var forestToCropArea = calculateArea(forestToCrop);

var forestToUrbanArea = calculateArea(forestToUrban);

var waterToCropArea = calculateArea(waterToCrop);


// ==========================================
// Print results
// ==========================================

print(
  "Forest → Cropland (km²)",
  forestToCropArea
);

print(
  "Forest → Urban (km²)",
  forestToUrbanArea
);

print(
  "Water → Cropland (km²)",
  waterToCropArea
);


// ==========================================
// Display transitions
// ==========================================

Map.addLayer(
  forestToCrop.selfMask(),
  {},
  "Forest → Cropland"
);

Map.addLayer(
  forestToUrban.selfMask(),
  {},
  "Forest → Urban"
);

Map.addLayer(
  waterToCrop.selfMask(),
  {},
  "Water → Cropland"
);


// ==========================================
// Center map
// ==========================================

Map.centerObject(studyArea, 11);
```

---

# 18. Connect to Previous Lessons

This project combines several things you've already learned:

```text
Project 12
Land cover
      ↓
Project 13
Built-up extraction
      ↓
Project 20
Land-cover change
      ↓
Project 38
Urban expansion
      ↓
Project 39
Land-cover transition matrix
```

The progression is important:

```text
What is there?
      ↓
What changed?
      ↓
Where did it change?
      ↓
What did it change INTO?
      ↓
How much area underwent each transition?
```

That's a much more powerful type of environmental analysis.

---

# 19. Things to Remember

### 1. Compare the same pixel location

We're comparing:

```text
2020 pixel
    ↓
same location
    ↓
2021 pixel
```

### 2. A transition has a direction

These are different:

```text
Forest → Urban
Urban → Forest
```

The first year is always the **starting class**.

### 3. Land-cover classes are categorical

Don't interpret:

```text
50 - 40 = 10
```

as a meaningful physical quantity.

Instead, use logical conditions:

```javascript
lc2020.eq(40).and(lc2021.eq(50))
```

### 4. `pixelArea()` gives area per pixel

Then:

```text
sum
↓
total area
```

### 5. A matrix summarizes many transitions

Instead of one change:

```text
Forest → Urban
```

we can eventually summarize:

```text
Forest → Crop
Forest → Urban
Crop → Urban
Water → Crop
...
```

---

# 20. Common Beginner Mistakes

### ❌ Mistake 1 — Comparing class numbers directly

Don't assume:

```javascript
lc2021.subtract(lc2020)
```

is a meaningful land-cover change matrix.

The class numbers are labels, not measurements.

---

### ❌ Mistake 2 — Reversing the years

This:

```javascript
lc2020.eq(10).and(lc2021.eq(50))
```

means:

> **Tree cover → Built-up**

But this:

```javascript
lc2020.eq(50).and(lc2021.eq(10))
```

means:

> **Built-up → Tree cover**

The order matters.

---

### ❌ Mistake 3 — Forgetting area units

`pixelArea()` produces square meters.

To convert to km²:

```javascript
.divide(1e6)
```

---

### ❌ Mistake 4 — Assuming every transition is real

Land-cover products contain classification errors.

A detected:

```text
Forest → Urban
```

transition is a **mapped change**, not automatically proof that actual construction occurred.

For research, you would validate important transitions using additional imagery or reference data.

---

# 21. Practice Project — Build Your Own Change Matrix

Now try expanding the code.

### Task 1

Add these transitions:

```text
Cropland → Urban
Grassland → Cropland
Cropland → Tree cover
Water → Urban
```

Use the class codes:

```text
Tree cover = 10
Grassland = 30
Cropland = 40
Built-up = 50
Water = 80
```

### Task 2

Calculate the area of each transition in km².

### Task 3

Display each transition on the map.

### Task 4 — Matrix thinking

Create a table like:

| Transition            | Area (km²) |
| --------------------- | ---------: |
| Tree cover → Cropland |        ... |
| Tree cover → Urban    |        ... |
| Cropland → Urban      |        ... |
| Grassland → Cropland  |        ... |
| Water → Cropland      |        ... |

The goal is to start thinking about land-cover change as **transitions between categories**, rather than just differences between two maps.

---

# 22. Conceptual Questions

1. What is a land-cover change matrix?
2. Why do we compare the same pixel location in two years?
3. What does this mean?

```javascript
lc2020.eq(10).and(lc2021.eq(50))
```

4. Why can't we simply subtract the 2020 and 2021 land-cover class numbers?
5. What is the difference between:

   * Forest → Urban
   * Urban → Forest
6. Why do we use `pixelArea()`?
7. What do the **rows** and **columns** of a transition matrix represent?
8. Why is a change matrix more informative than simply calculating the total changed area?

### Key idea

> **A land-cover change matrix doesn't just tell us that land changed. It tells us the direction of the transition — what each pixel was before, what it became, and how much area underwent that transition.**
